# Saliency Visualization

Scalp maps of which bipolar channels NeuroGATE relies on, computed with Input × Gradient saliency on the evaluation set.
Each channel's score is split between its two electrodes, then drawn as a topomap for the most confident correctly classified Abnormal and Normal recordings.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon, Wedge
from matplotlib.ticker import ScalarFormatter
import mne
import os
import gc

from cereprocess.train.xloop import get_datanpz
from cereprocess.train.misc import def_dev
from cereprocess.datasets.defaults import get_def_ds
from cereprocess.datasets.pipeline import general_pipeline
from cereprocess.datasets.pytordataset import EEGDataset
from cereprocess.train.train import evaluate
from cereprocess.train.callbacks import def_metrics, History
from models.neurogate import NeuroGATE

# Data

In [ ]:
device = def_dev()
mins = 10
input_size = (22, mins * 60 * 50)
tuh, nmt, nmt_4k = get_def_ds(mins)

In [ ]:
# Change the dataset currently in use from over here
data_path = "path/to/nmt_scalp_eeg_dataset"
curr_data = (data_path, *nmt[1:])
data_dir, data_description = get_datanpz(curr_data[0], curr_data[3], general_pipeline(dataset='NMT', length_minutes=mins, min_len=0, max_len=5000), input_size)
eval_loader = DataLoader(EEGDataset(os.path.join(data_dir, 'eval')), batch_size=1, shuffle=False)

# Load a trained model

In [ ]:
torch.cuda.empty_cache()
gc.collect()

model = NeuroGATE().to(device)
model.load_state_dict(torch.load("results/nmt/models/model_XX.pt", map_location=device))
metrics = def_metrics(device)
history = History()
criterion = nn.CrossEntropyLoss()
evaluate(model, eval_loader, criterion, device, metrics, history)
print({key: value[-1] if isinstance(value, list) else value for key, value in history.history['val'].items()})

# Saliency topomaps

In [ ]:
# Publication style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'seaborn-whitegrid')
plt.rcParams.update({
    'font.size': 18,
    'axes.labelsize': 18,
    'figure.dpi': 300,
})

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
CHANNEL_PAIRS = [
    ('FP1', 'F7'), ('FP1', 'F3'), ('FP2', 'F4'), ('FP2', 'F8'),
    ('F7', 'T3'),  ('F3', 'C3'),  ('F4', 'C4'),  ('F8', 'T4'),
    ('A1', 'T3'),  ('T3', 'C3'),  ('C3', 'CZ'),  ('CZ', 'C4'),
    ('C4', 'T4'),  ('T4', 'A2'),  ('T3', 'T5'),  ('C3', 'P3'),
    ('C4', 'P4'),  ('T4', 'T6'),  ('T5', 'O1'),  ('P3', 'O1'),
    ('P4', 'O2'),  ('T6', 'O2'),
]

POS = {
    'FZ': (0.0, 0.45),  'CZ': (0.0, 0.0),   'PZ': (0.0, -0.45),
    'FP1': (-0.30, 0.85),'FP2': (0.30, 0.85),
    'F3': (-0.35, 0.45), 'F4': (0.35, 0.45),
    'F7': (-0.75, 0.55), 'F8': (0.75, 0.55),
    'A1': (-1.15, 0.0), 'T3': (-0.85, 0.0), 'C3': (-0.45, 0.0), 'C4': (0.45, 0.0), 'T4': (0.85, 0.0), 'A2': (1.15, 0.0),
    'T5': (-0.75, -0.55),'T6': (0.75, -0.55),
    'P3': (-0.35, -0.45),'P4': (0.35, -0.45),
    'O1': (-0.30, -0.85),'O2': (0.30, -0.85),
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_top_samples(model, loader, device, k=10):
    print("Scanning for top confidence samples...")
    model.eval()
    normal_samples, seizure_samples = [], []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            targets = labels.argmax(dim=1) if labels.ndim > 1 else labels
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            confs, preds = probs.max(dim=1)
            
            for i in range(len(preds)):
                if preds[i] == targets[i]: 
                    data = (confs[i].item(), inputs[i].detach().cpu())
                    if preds[i] == 0: normal_samples.append(data)
                    else: seizure_samples.append(data)
    
    normal_samples.sort(key=lambda x: x[0], reverse=True)
    seizure_samples.sort(key=lambda x: x[0], reverse=True)
    return normal_samples, seizure_samples

def compute_saliency_map(model, input_tensor, target_class, device):
    """Computes Input * Gradient importance."""
    model.eval()
    input_tensor = input_tensor.unsqueeze(0).to(device)
    input_tensor.requires_grad = True
    
    output = model(input_tensor)
    score = output[0, target_class]
    
    model.zero_grad()
    score.backward()
    
    # Input * Gradient
    grads = input_tensor.grad
    saliency = (grads * input_tensor).abs().squeeze().mean(dim=1).detach().cpu().numpy()
    
    node_scores = {name: 0.0 for name in POS.keys()}
    counts = {name: 0 for name in POS.keys()}
    
    for idx, (ch1, ch2) in enumerate(CHANNEL_PAIRS):
        if idx >= len(saliency): break
        s = saliency[idx]
        if ch1 in node_scores: node_scores[ch1] += s; counts[ch1] += 1
        if ch2 in node_scores: node_scores[ch2] += s; counts[ch2] += 1
            
    for name in node_scores:
        if counts[name] > 0: node_scores[name] /= counts[name]
            
    return node_scores

def draw_single_head(ax, node_scores, global_vmax, title):
    names = list(node_scores.keys())
    data = np.array([node_scores[n] for n in names])
    
    pos_plot = np.array([POS[n] for n in names])
    for i, name in enumerate(names):
        scale = 0.95 if name in ['A1', 'A2'] else 0.90
        pos_plot[i] *= scale

    HEAD_RADIUS = 1.0

    # Heatmap
    im, _ = mne.viz.plot_topomap(data, pos_plot, axes=ax, show=False, cmap='Reds', contours=0,
                                 sensors=False, extrapolate='box', sphere=(0,0,0, HEAD_RADIUS),
                                 vlim=(0, global_vmax))

    # Graphics
    # White "Donut" mask to clean up edges
    ax.add_patch(Wedge((0, 0), r=3.0, theta1=0, theta2=360, width=2.0, color='white', zorder=2))
    ax.add_patch(Circle((0, 0), radius=HEAD_RADIUS, color='black', linewidth=2, fill=False, zorder=3))
    ax.add_patch(Polygon([(-0.1, 0.99), (0, 1.1), (0.1, 0.99)], color='black', fill=False, linewidth=2, zorder=3))
    ax.add_patch(Wedge((-1.0, 0.0), r=0.15, theta1=90, theta2=270, width=0.05, color='black', zorder=3))
    ax.add_patch(Wedge((1.0, 0.0), r=0.15, theta1=-90, theta2=90, width=0.05, color='black', zorder=3))

    # Badges
    for idx, name in enumerate(names):
        x, y = pos_plot[idx]
        ax.add_patch(Circle((x, y), radius=0.085, color='white', zorder=4))
        ax.add_patch(Circle((x, y), radius=0.085, fill=False, edgecolor='black', linewidth=1, zorder=5))
        ax.text(x, y, name, fontsize=10, ha='center', va='center', fontweight='bold', color='black', zorder=6)

    ax.set_title(title, pad=15)
    
    # ZOOM IN: Tight limits make the head fill the subplot
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal')
    ax.axis('off')
    
    return im

# ==========================================
# EXECUTION
# ==========================================

# 1. Get Samples
normals, seizures = get_top_samples(model, eval_loader, device, k=10)

# 2. Select Specific Samples
selected_data = [
    ('Abnormal', seizures[0][0], seizures[0][1], 1),
    ('Abnormal', seizures[4][0], seizures[4][1], 1),
    ('Normal',  normals[0][0],  normals[0][1],  0)
]

# 3. Compute Maps & Global Max
maps = []
max_vals = []
for label, conf, tensor, cls_idx in selected_data:
    s_map = compute_saliency_map(model, tensor, cls_idx, device)
    maps.append((label, conf, s_map))
    max_vals.append(max(s_map.values()))

global_vmax = max(max_vals)
print(f"Global Max Saliency: {global_vmax:.2e}")

# 4. Plot (Wider figure size for horizontal layout)
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Adjust spacing to bring them closer/further as needed
plt.subplots_adjust(wspace=0.1)

for i, (label, conf, s_map) in enumerate(maps):
    im = draw_single_head(axes[i], s_map, global_vmax, f"{label}")

# Shared Colorbar
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.03)
cbar.set_label('Saliency Magnitude (Input x Gradient)', rotation=270, labelpad=25)
cbar.ax.tick_params(labelsize=12)

# Format scientific notation
formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0)) 
cbar.ax.yaxis.set_major_formatter(formatter)

plt.savefig("Saliency_Comparison.png", dpi=300, bbox_inches='tight')
plt.show()